# Arithmetic Transformer — JAX · Flax NNX · Optax

A ~10M-parameter decoder-only transformer trained from scratch on 3-digit integer addition.
No PyTorch. No HuggingFace. No pre-trained weights.

**Runtime**: TPU v2-8 (free Colab) or T4 GPU — same code, same weights, ~2 min either way  
**Stack**: JAX (XLA backend), Flax NNX (modern stateful API), Optax (optimizer + schedule)

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import numpy as np
import time

print(f"JAX {jax.__version__}  •  backend: {jax.default_backend()}")
print(f"devices: {jax.devices()}")

In [ ]:
# Fixed vocabulary: digits, space, operators, pad
CHARS = "0123456789 +="
VOCAB = list(CHARS) + ["<P>"]
C2I   = {c: i for i, c in enumerate(VOCAB)}
I2C   = {i: c for c, i in C2I.items()}
V     = len(VOCAB)   # 14
PAD   = C2I["<P>"]

# Longest possible sequence: "999 + 999 = 1998" = 16 chars → pad to 20
T = 20

def enc(s: str) -> list:
    return [C2I[c] for c in s]

def dec(ids) -> str:
    return "".join(I2C[i] for i in ids if i != PAD)

# Sanity check
ex = enc("123 + 456 = 579")
assert dec(ex) == "123 + 456 = 579"
print(f"vocab size: {V}  |  seq len: {T}  |  example: {ex}")

In [ ]:
rng = np.random.default_rng(0)

def make_seq(a: int, b: int) -> np.ndarray:
    s = f"{a} + {b} = {a + b}"
    ids = enc(s)
    return np.array(ids + [PAD] * (T - len(ids)), dtype=np.int32)

def sample_batch(size: int) -> jnp.ndarray:
    a = rng.integers(0, 1000, size)
    b = rng.integers(0, 1000, size)
    seqs = np.stack([make_seq(int(aa), int(bb)) for aa, bb in zip(a, b)])
    return jnp.array(seqs)

# Quick check: a batch of 4
batch = sample_batch(4)
for row in batch:
    print(" ", dec(row.tolist()))

In [ ]:
class Attention(nnx.Module):
    def __init__(self, d: int, h: int, rngs: nnx.Rngs):
        self.h  = h
        self.dh = d // h
        self.qkv  = nnx.Linear(d, 3 * d, rngs=rngs)
        self.proj = nnx.Linear(d, d,     rngs=rngs)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        B, L, D = x.shape
        # (B, L, 3*D) → (B, h, 3, L, dh)
        qkv = self.qkv(x).reshape(B, L, 3, self.h, self.dh).transpose(0, 3, 2, 1, 4)
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]  # each (B, h, L, dh)

        w = (q @ k.swapaxes(-2, -1)) * self.dh ** -0.5       # (B, h, L, L)
        w = jnp.where(jnp.tril(jnp.ones((L, L))), w, -1e9)  # causal mask
        w = jax.nn.softmax(w, axis=-1)

        y = (w @ v).transpose(0, 2, 1, 3).reshape(B, L, D)   # (B, L, D)
        return self.proj(y)


class Block(nnx.Module):
    def __init__(self, d: int, h: int, ff: int, rngs: nnx.Rngs):
        self.attn = Attention(d, h, rngs)
        self.fc1  = nnx.Linear(d, ff, rngs=rngs)
        self.fc2  = nnx.Linear(ff, d, rngs=rngs)
        self.ln1  = nnx.LayerNorm(d, rngs=rngs)
        self.ln2  = nnx.LayerNorm(d, rngs=rngs)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        x = x + self.attn(self.ln1(x))
        x = x + self.fc2(jax.nn.gelu(self.fc1(self.ln2(x))))
        return x


class GPT(nnx.Module):
    def __init__(self, v: int, t: int, d: int, h: int, ff: int, nl: int, rngs: nnx.Rngs):
        self.tok    = nnx.Embed(v, d, rngs=rngs)
        self.pos    = nnx.Embed(t, d, rngs=rngs)
        self.blocks = [Block(d, h, ff, rngs) for _ in range(nl)]
        self.ln     = nnx.LayerNorm(d, rngs=rngs)
        self.head   = nnx.Linear(d, v, use_bias=False, rngs=rngs)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        B, L = x.shape
        h = self.tok(x) + self.pos(jnp.arange(L))  # (B, L, d)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln(h))                # (B, L, V)

In [ ]:
# ~10.7M parameters: 6 layers × 12 × 384² ≈ 10.6M + embeddings
D, H, FF, NL = 384, 6, 1536, 6

model = GPT(V, T, D, H, FF, NL, rngs=nnx.Rngs(0))

_, state = nnx.split(model)
n_params = sum(v.size for v in jax.tree_util.tree_leaves(state))
print(f"Parameters: {n_params / 1e6:.2f}M")

# On TPU v2-8: wrap in jax.pmap over 8 cores for ~8x throughput.
# On a single T4 or A4000, nnx.jit below is sufficient.

In [ ]:
STEPS   = 15_000
BATCH   = 512
LR_PEAK = 3e-4

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=LR_PEAK,
    warmup_steps=500, decay_steps=STEPS,
)
tx = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(schedule, weight_decay=0.1),
)
optimizer = nnx.Optimizer(model, tx)


@nnx.jit
def train_step(model: GPT, opt: nnx.Optimizer, seqs: jnp.ndarray) -> jnp.ndarray:
    x  = seqs[:, :-1]                           # (B, T-1) — input tokens
    y  = seqs[:, 1:]                            # (B, T-1) — targets
    npad = (y != PAD).astype(jnp.float32)       # mask out PAD positions in loss

    def loss_fn(model):
        logits = model(x)                       # (B, T-1, V)
        ce = optax.softmax_cross_entropy_with_integer_labels(logits, y)
        return (ce * npad).sum() / npad.sum()

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    opt.update(grads)
    return loss

In [ ]:
losses = []
t0 = time.perf_counter()

for step in range(1, STEPS + 1):
    loss = train_step(model, optimizer, sample_batch(BATCH))
    losses.append(float(loss))

    if step % 1_000 == 0:
        elapsed = time.perf_counter() - t0
        tok_s   = step * BATCH * (T - 1) / elapsed
        print(f"step {step:5d}/{STEPS}  loss {loss:.4f}  {tok_s / 1e3:.1f}k tok/s")

print(f"\ntotal wall time: {time.perf_counter() - t0:.0f}s")

In [ ]:
import matplotlib.pyplot as plt

smooth = np.convolve(losses, np.ones(200) / 200, mode="valid")
plt.figure(figsize=(9, 3))
plt.plot(losses, alpha=0.2, color="steelblue")
plt.plot(smooth, color="steelblue")
plt.xlabel("step")
plt.ylabel("cross-entropy loss")
plt.title(f"{n_params/1e6:.1f}M-param GPT  ·  3-digit addition  ·  {BATCH} batch  ·  {D}d {NL}L")
plt.tight_layout()
plt.show()

In [ ]:
def predict(model: GPT, a: int, b: int) -> str:
    """Autoregressively decode the answer to `a + b`."""
    ctx = enc(f"{a} + {b} = ")
    for _ in range(6):                          # max answer: 4 digits + margin
        pad  = [PAD] * max(0, T - 1 - len(ctx))
        x    = jnp.array((ctx + pad)[: T - 1])[None]  # (1, T-1)
        logits = model(x)                       # (1, T-1, V)
        pos  = min(len(ctx) - 1, T - 2)
        nxt  = int(jnp.argmax(logits[0, pos]))
        if nxt == PAD or len(ctx) >= T - 1:
            break
        ctx.append(nxt)
    ans_start = len(enc(f"{a} + {b} = "))
    return "".join(I2C[t] for t in ctx[ans_start:])


def accuracy(max_val: int, n: int = 500) -> float:
    a = rng.integers(0, max_val + 1, n)
    b = rng.integers(0, max_val + 1, n)
    return sum(
        predict(model, int(aa), int(bb)) == str(int(aa) + int(bb))
        for aa, bb in zip(a, b)
    ) / n


print("Accuracy by operand range (n=500 random pairs each):")
for label, mx in [("1-digit  (0–9)", 9), ("2-digit (0–99)", 99), ("3-digit (0–999)", 999)]:
    print(f"  {label}: {accuracy(mx) * 100:.1f}%")

print("\nSample predictions:")
for a, b in [(3, 7), (47, 53), (123, 456), (999, 1), (999, 999)]:
    pred  = predict(model, a, b)
    mark  = "✓" if pred == str(a + b) else "✗"
    print(f"  {a:3d} + {b:3d} = {pred:>5}   (expected {a + b})  {mark}")